In [17]:
#Step 1: Import required libraries
import re
from collections import Counter
import nltk
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
# Download required NLTK resources
nltk.download('punkt')
nltk.download('punkt_tab')


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [20]:
# Step 2: Create the healthcare article corpus
# ============================================================
articles = {
"Diabetes": """
Diabetes is a chronic health condition that affects blood sugar levels.
A healthy diet, regular physical activity, and proper medication can help
manage diabetes. People with diabetes should monitor blood glucose levels
regularly and follow a balanced nutrition plan. Early diagnosis and
preventive healthcare can reduce the risk of diabetes complications.
""",
"Heart Health": """
Heart health is important for maintaining a healthy lifestyle.
Regular exercise, a balanced diet, and healthy blood pressure can reduce
the risk of heart disease. Eating fruits, vegetables, and whole grains
supports cardiovascular health. Avoiding smoking and managing cholesterol
levels can also protect the heart.
""",
"Nutrition": """
Good nutrition provides the body with essential nutrients for healthy
growth and development. A balanced diet should include fruits, vegetables,
proteins, whole grains, vitamins, and minerals. Healthy eating habits
can improve energy levels, support the immune system, and reduce the risk
of chronic diseases.
""",
"Mental Wellness": """
Mental wellness is an important part of overall health. Regular exercise,
adequate sleep, relaxation, and social support can improve mental health.
Stress management techniques such as meditation and mindfulness can help
people manage anxiety and maintain emotional wellbeing. Seeking professional
support is important when mental health problems affect daily life.
""",
"Preventive Healthcare": """
Preventive healthcare focuses on preventing diseases before they become
serious. Regular health checkups, vaccination, screening tests, healthy
eating, physical activity, and adequate sleep are important preventive
measures. Early detection of health problems can support timely treatment
and improve overall health outcomes.
"""
}

In [23]:
# ============================================================
# Step 3: Text preprocessing function
# ============================================================
def preprocess(text):
  # Convert text to lowercase
  text = text.lower()
  # Keep only alphabetic words
  text = re.sub(r'[^a-z\s]', '', text)
  # Tokenize the text
  tokens = word_tokenize(text)
  return tokens
  # ============================================================
# Step 4: Calculate lexical diversity
# ============================================================
def lexical_diversity(tokens):
  if len(tokens) == 0:
    return 0
  unique_words = set(tokens)#remove duplicate words
  return len(unique_words) / len(tokens)
# ============================================================
# Step 5: Generate n-grams
# ============================================================
def generate_ngrams(tokens, n):
  return [
  ' '.join(tokens[i:i+n])
  for i in range(len(tokens) - n + 1)
  ]


In [26]:
article_tokens = {}
for topic, article in articles.items():
  tokens = preprocess(article)
  article_tokens[topic] = tokens
  # Calculate lexical diversity
  diversity = lexical_diversity(tokens)
  # Generate unigrams
  unigrams = generate_ngrams(tokens, 1)
  # Generate bigrams
  bigrams = generate_ngrams(tokens, 2)
  # Generate trigrams
  trigrams = generate_ngrams(tokens, 3)
  # Count frequent n-grams
  unigram_freq = Counter(unigrams)
  bigram_freq = Counter(bigrams)
  trigram_freq = Counter(trigrams)
  print("\n" + "=" * 70)
  print("TOPIC:", topic)
  print("=" * 70)
  print("\n1. Lexical Diversity")
  print("Total words :", len(tokens))
  print("Unique words :", len(set(tokens)))
  print("Lexical diversity:", round(diversity, 3))
  print("\n2. Top 5 Unigrams")
  print(unigram_freq.most_common(5))
  print("\n3. Top 5 Bigrams")
  print(bigram_freq.most_common(5))
  print("\n4. Top 5 Trigrams")
  print(trigram_freq.most_common(5))



TOPIC: Diabetes

1. Lexical Diversity
Total words : 51
Unique words : 41
Lexical diversity: 0.804

2. Top 5 Unigrams
[('diabetes', 4), ('a', 3), ('and', 3), ('blood', 2), ('levels', 2)]

3. Top 5 Bigrams
[('diabetes is', 1), ('is a', 1), ('a chronic', 1), ('chronic health', 1), ('health condition', 1)]

4. Top 5 Trigrams
[('diabetes is a', 1), ('is a chronic', 1), ('a chronic health', 1), ('chronic health condition', 1), ('health condition that', 1)]

TOPIC: Heart Health

1. Lexical Diversity
Total words : 45
Unique words : 36
Lexical diversity: 0.8

2. Top 5 Unigrams
[('heart', 3), ('and', 3), ('health', 2), ('a', 2), ('healthy', 2)]

3. Top 5 Bigrams
[('heart health', 1), ('health is', 1), ('is important', 1), ('important for', 1), ('for maintaining', 1)]

4. Top 5 Trigrams
[('heart health is', 1), ('health is important', 1), ('is important for', 1), ('important for maintaining', 1), ('for maintaining a', 1)]

TOPIC: Nutrition

1. Lexical Diversity
Total words : 44
Unique words : 39

In [30]:
topics = list(articles.keys())
documents = list(articles.values())
vectorizer = TfidfVectorizer(
stop_words='english',
ngram_range=(1, 3) #considers unigrams, bigrams, and trigrams
)
tfidf_matrix = vectorizer.fit_transform(documents)
feature_names = vectorizer.get_feature_names_out()


In [32]:
print("\n\n")
print("=" * 70)
print("TOP 5 TF-IDF KEYWORDS FOR EACH HEALTHCARE ARTICLE")
print("=" * 70)
top_keywords = {}
for i, topic in enumerate(topics):
  # Get TF-IDF scores for this article
  scores = tfidf_matrix[i].toarray().flatten()
  # Sort indexes according to TF-IDF score
  sorted_indexes = scores.argsort()[::-1]
top_5 = []
for index in sorted_indexes[:5]:
  top_5.append(
  (feature_names[index], round(scores[index], 3))
  )
  top_keywords[topic] = top_5
  print("\n", topic)
  print("-" * 40)
for keyword, score in top_5:
  print(keyword, ":", score)





TOP 5 TF-IDF KEYWORDS FOR EACH HEALTHCARE ARTICLE

 Preventive Healthcare
----------------------------------------

 Preventive Healthcare
----------------------------------------

 Preventive Healthcare
----------------------------------------

 Preventive Healthcare
----------------------------------------

 Preventive Healthcare
----------------------------------------
health : 0.187
preventive : 0.179
activity adequate sleep : 0.111
activity adequate : 0.111
support timely treatment : 0.111


In [33]:
print("\n\n")
print("=" * 70)
print("COMPARISON OF HEALTHCARE TOPICS")
print("=" * 70)
print("\n{:<25} {:<15} {:<15}".format(
"Topic", "Total Words", "Lexical Diversity"
))
print("-" * 55)
for topic in topics:
  tokens = article_tokens[topic]
  diversity = lexical_diversity(tokens)
  print("{:<25} {:<15} {:.3f}".format(
  topic,
  len(tokens),
  diversity
  ))





COMPARISON OF HEALTHCARE TOPICS

Topic                     Total Words     Lexical Diversity
-------------------------------------------------------
Diabetes                  51              0.804
Heart Health              45              0.800
Nutrition                 44              0.886
Mental Wellness           50              0.800
Preventive Healthcare     41              0.902
